# 07 — Trip Duration Prediction Agent

**Type:** Side project  
**Purpose:** Train a trip-duration regression model on the Silver taxi data, then wire it to an LLM agent that accepts natural-language pickup/dropoff requests, fetches live weather and current time context, and returns a plain-English duration estimate.

---

### Architecture

```
User: "I'm at Times Square, taxi to JFK please"
         │
         ▼
   LLM Agent (Llama 3.3 70B)
         │
   ┌─────┴──────────────────────────────┐
   │  Tool calls (function calling)    │
   │  1. get_datetime_features()        │  ← current NYC time
   │  2. geocode_address(pickup)        │  ← lat/lon via Nominatim
   │  3. geocode_address(dropoff)       │  ← lat/lon via Nominatim
   │  4. get_weather(lat, lon)          │  ← Open-Meteo API
   │  5. predict_duration(...)          │  ← trained HistGBT model
   └────────────────────────────────────┘
         │
         ▼
   "Based on current conditions (3pm Thursday, light rain),
    your trip should take approximately 42 minutes (~12.4 miles)."
```

### Key design decisions

| Decision | Rationale |
|----------|-----------|
| **Duration, not wait time** | Dataset only has pickup→dropoff events, not order→pickup. Trip duration is the only well-supported prediction target. |
| **Same features as fare model** | `ML_FEATURE_COLUMNS` (location, hour, day, distance, passengers, rate code) are all known at ride start. |
| **Haversine × 1.3 for distance** | At inference we only have coordinates, not measured odometer distance. Road distance ≈ straight-line × 1.3 for NYC. |
| **Live tools for context** | Weather and exact time are fetched at query time so predictions reflect current conditions. |
| **Separate model from fare model** | Duration and fare have different targets and distributions; separate training gives cleaner MLflow tracking. |

## Setup

In [ ]:
%pip install requests mlflow pytz --quiet

In [ ]:
%restart_python

In [ ]:
import importlib
import json
import math

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import src.constants
import src.transforms
from mlflow.deployments import get_deploy_client
from pyspark.sql import functions as F
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

importlib.reload(src.constants)
importlib.reload(src.transforms)

from src.constants import (  # noqa: E402
    AGENT_LLM_ENDPOINT,
    DURATION_MODEL_NAME,
    LAT_LON_BIN_SIZE,
    ML_DURATION_TARGET_COLUMN,
    ML_FEATURE_COLUMNS,
    ML_RANDOM_STATE,
    ML_SAMPLE_FRACTION,
    ML_TEST_SIZE,
    SILVER_TABLE,
)

deploy_client = get_deploy_client("databricks")
print("Setup complete")
print(f"  Duration target : {ML_DURATION_TARGET_COLUMN}")
print(f"  Model name      : {DURATION_MODEL_NAME}")
print(f"  LLM endpoint    : {AGENT_LLM_ENDPOINT}")

## Duration Model — Build Feature Table

Reuses the same Silver features as the fare model (`ML_FEATURE_COLUMNS`) but predicts `trip_duration_min` instead of `total_amount`.  
Both models are structurally identical — the only change is the target column.

In [ ]:
# ── Read Silver & select features + duration target ───────────────────────────
silver_df = spark.read.table(SILVER_TABLE)
feature_df = silver_df.select(*ML_FEATURE_COLUMNS, ML_DURATION_TARGET_COLUMN)

# Drop nulls (rate_code_id NULLs from code-99 recovery, GPS-nulled rows)
feature_df = feature_df.dropna()

# Drop rows with empty zone strings (GPS coords outside NYC bbox)
feature_df = feature_df.filter(
    (F.col("pickup_zone") != "") & (F.col("dropoff_zone") != "")
)

# Drop zero/negative durations (data quality guard)
feature_df = feature_df.filter(F.col(ML_DURATION_TARGET_COLUMN) > 0)
print(f"Rows after filtering: {feature_df.count():,}")

# ── Parse zone strings into numeric lat/lon bins ──────────────────────────────
# Zone format: "lat_bin,lon_bin" (e.g. "40.75,-73.99")
for prefix in ["pickup", "dropoff"]:
    feature_df = feature_df.withColumn(
        f"{prefix}_lat_bin",
        F.split(F.col(f"{prefix}_zone"), ",")[0].cast("double"),
    ).withColumn(
        f"{prefix}_lon_bin",
        F.split(F.col(f"{prefix}_zone"), ",")[1].cast("double"),
    )
feature_df = feature_df.drop("pickup_zone", "dropoff_zone")

# Cast is_weekend boolean → int
feature_df = feature_df.withColumn("is_weekend", F.col("is_weekend").cast("int"))

# ── Sample (same fraction as fare model) ─────────────────────────────────────
feature_df = feature_df.sample(fraction=ML_SAMPLE_FRACTION, seed=ML_RANDOM_STATE)
pdf = feature_df.toPandas()

# ── One-hot encode rate_code_id ───────────────────────────────────────────────
pdf = pd.get_dummies(pdf, columns=["rate_code_id"], prefix="rc", dtype=int)

print(f"Feature table shape: {pdf.shape}")
print(
    f"Target mean: {pdf[ML_DURATION_TARGET_COLUMN].mean():.1f} min  "
    f"| std: {pdf[ML_DURATION_TARGET_COLUMN].std():.1f} min"
)

X = pdf.drop(columns=[ML_DURATION_TARGET_COLUMN])
y = pdf[ML_DURATION_TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=ML_TEST_SIZE, random_state=ML_RANDOM_STATE
)
print(f"Train: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]:,} rows × {X_test.shape[1]} features")

## Duration Model — Train & Register

In [ ]:
# ── Train HistGradientBoostingRegressor (same hyperparams as fare model) ──────
dur_params = {
    "max_iter": 300,
    "max_depth": 8,
    "learning_rate": 0.05,
    "min_samples_leaf": 50,
    "random_state": ML_RANDOM_STATE,
}

duration_model = HistGradientBoostingRegressor(**dur_params)
duration_model.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred = duration_model.predict(X_test)
dur_metrics = {
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
    "mae": float(mean_absolute_error(y_test, y_pred)),
    "r2": float(r2_score(y_test, y_pred)),
}

# ── Log & register to MLflow ──────────────────────────────────────────────────
# input_example lets MLflow auto-infer the signature (required for Unity Catalog)
input_example = X_train.iloc[:5]

with mlflow.start_run(run_name="duration_HistGradientBoosting"):
    mlflow.log_params(dur_params)
    mlflow.log_param("model_type", "HistGradientBoostingRegressor")
    mlflow.log_param("target", ML_DURATION_TARGET_COLUMN)
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_rows", X_train.shape[0])
    mlflow.log_param("test_rows", X_test.shape[0])
    mlflow.log_metrics(dur_metrics)
    mlflow.sklearn.log_model(
        duration_model,
        name="model",
        input_example=input_example,
        registered_model_name=DURATION_MODEL_NAME,
    )
    dur_run_id = mlflow.active_run().info.run_id

# Store feature column order — used by predict_duration() to align inference rows
duration_feature_cols = list(X_train.columns)

print("Trip Duration Model — HistGradientBoosting")
print("=" * 45)
print(f"  RMSE:  {dur_metrics['rmse']:.2f} min")
print(f"  MAE:   {dur_metrics['mae']:.2f} min")
print(f"  R\u00b2:    {dur_metrics['r2']:.4f}")
print(f"\nMLflow run ID  : {dur_run_id}")
print(f"Registered as  : {DURATION_MODEL_NAME}")

## Agent Tools

Four tools that the LLM can call during a conversation:

| Tool | Data source | Notes |
|------|-------------|-------|
| `get_datetime_features` | System clock | Current NYC hour/day for model input |
| `geocode_address` | Nominatim (OSM) | Free, no API key required |
| `get_weather` | Open-Meteo | Free, no API key required |
| `predict_duration` | Trained HistGBT model | Uses haversine × 1.3 for road-distance approximation |

In [ ]:
import requests
import pytz
from datetime import datetime

# Spark dayofweek convention: 1=Sun, 2=Mon, ..., 7=Sat
# Python weekday():           0=Mon, 1=Tue, ..., 6=Sun
_PYTHON_TO_SPARK_DOW = {0: 2, 1: 3, 2: 4, 3: 5, 4: 6, 5: 7, 6: 1}


def get_datetime_features() -> dict:
    """Return current NYC time as model features: hour_of_day, day_of_week, is_weekend."""
    now = datetime.now(pytz.timezone("America/New_York"))
    python_dow = now.weekday()
    return {
        "hour_of_day": now.hour,
        "day_of_week": _PYTHON_TO_SPARK_DOW[python_dow],
        "is_weekend": 1 if python_dow >= 5 else 0,
        "day_name": now.strftime("%A"),
        "current_time": now.strftime("%Y-%m-%d %H:%M %Z"),
    }


def geocode_address(address: str) -> dict:
    """Geocode an NYC address or landmark to lat/lon using Nominatim (OpenStreetMap)."""
    resp = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": f"{address}, New York City", "format": "json", "limit": 1},
        headers={"User-Agent": "nyc-taxi-duration-agent/1.0"},
        timeout=10,
    )
    resp.raise_for_status()
    results = resp.json()
    if not results:
        raise ValueError(f"Could not geocode address: {address!r}")
    return {
        "lat": float(results[0]["lat"]),
        "lon": float(results[0]["lon"]),
        "display_name": results[0]["display_name"],
    }


def get_weather(lat: float, lon: float) -> dict:
    """Get current weather at a lat/lon location using the Open-Meteo API."""
    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,precipitation,cloud_cover,wind_speed_10m",
            "temperature_unit": "fahrenheit",
            "wind_speed_unit": "mph",
            "precipitation_unit": "inch",
            "timezone": "America/New_York",
        },
        timeout=10,
    )
    resp.raise_for_status()
    c = resp.json()["current"]
    return {
        "temperature_f": c["temperature_2m"],
        "precipitation_inch": c["precipitation"],
        "cloud_cover_pct": c["cloud_cover"],
        "wind_speed_mph": c["wind_speed_10m"],
    }


def predict_duration(
    pickup_lat: float,
    pickup_lon: float,
    dropoff_lat: float,
    dropoff_lon: float,
    hour_of_day: int,
    day_of_week: int,
    is_weekend: int,
    passenger_count: int = 1,
    rate_code_id: int = 1,
) -> dict:
    """Predict NYC taxi trip duration in minutes using the trained HistGBT model.

    trip_distance is approximated as haversine straight-line distance * 1.3
    (NYC road distance ~30% longer than straight line on average).
    """
    bin_size = LAT_LON_BIN_SIZE  # 0.01 degrees

    # Bin coordinates to the same 0.01-degree grid used during training
    pickup_lat_bin = round(pickup_lat / bin_size) * bin_size
    pickup_lon_bin = round(pickup_lon / bin_size) * bin_size
    dropoff_lat_bin = round(dropoff_lat / bin_size) * bin_size
    dropoff_lon_bin = round(dropoff_lon / bin_size) * bin_size

    # Haversine straight-line distance (miles) * 1.3 road-distance correction
    R = 3958.8
    lat1, lon1 = math.radians(pickup_lat), math.radians(pickup_lon)
    lat2, lon2 = math.radians(dropoff_lat), math.radians(dropoff_lon)
    a = (
        math.sin((lat2 - lat1) / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    )
    trip_distance = round(2 * R * math.asin(math.sqrt(a)) * 1.3, 2)

    # One-hot encode rate_code_id (rc_1 … rc_6)
    rc_cols = {f"rc_{i}": (1 if rate_code_id == i else 0) for i in range(1, 7)}

    row = {
        "hour_of_day": hour_of_day,
        "day_of_week": day_of_week,
        "is_weekend": is_weekend,
        "trip_distance": trip_distance,
        "passenger_count": passenger_count,
        "pickup_lat_bin": pickup_lat_bin,
        "pickup_lon_bin": pickup_lon_bin,
        "dropoff_lat_bin": dropoff_lat_bin,
        "dropoff_lon_bin": dropoff_lon_bin,
        **rc_cols,
    }
    features = pd.DataFrame([row])

    # Align to training column order; fill any unseen OHE columns with 0
    for col in duration_feature_cols:
        if col not in features.columns:
            features[col] = 0
    features = features[duration_feature_cols]

    predicted_min = max(1.0, float(duration_model.predict(features)[0]))
    return {
        "predicted_duration_min": round(predicted_min, 1),
        "estimated_road_distance_miles": trip_distance,
    }


print(
    "Tools defined: get_datetime_features, geocode_address, get_weather, predict_duration"
)

## Trip Duration Agent

The agent uses Databricks Foundation Model API (OpenAI-compatible) with tool/function calling.  
The LLM orchestrates all tool calls autonomously — the user just asks in plain English.

In [ ]:
_TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_datetime_features",
            "description": "Get the current NYC date and time as model features (hour_of_day, day_of_week, is_weekend). Always call this first.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "geocode_address",
            "description": "Convert an NYC street address or landmark name to latitude/longitude coordinates.",
            "parameters": {
                "type": "object",
                "properties": {
                    "address": {
                        "type": "string",
                        "description": "Street address or NYC landmark (e.g. 'Times Square', '34th St Penn Station', 'JFK Airport')",
                    }
                },
                "required": ["address"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather conditions at a location using latitude and longitude.",
            "parameters": {
                "type": "object",
                "properties": {
                    "lat": {
                        "type": "number",
                        "description": "Latitude of the location",
                    },
                    "lon": {
                        "type": "number",
                        "description": "Longitude of the location",
                    },
                },
                "required": ["lat", "lon"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "predict_duration",
            "description": "Predict NYC taxi trip duration in minutes using a trained ML model. Call this after collecting all other inputs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pickup_lat": {"type": "number", "description": "Pickup latitude"},
                    "pickup_lon": {"type": "number", "description": "Pickup longitude"},
                    "dropoff_lat": {
                        "type": "number",
                        "description": "Dropoff latitude",
                    },
                    "dropoff_lon": {
                        "type": "number",
                        "description": "Dropoff longitude",
                    },
                    "hour_of_day": {
                        "type": "integer",
                        "description": "Current hour (0\u201323)",
                    },
                    "day_of_week": {
                        "type": "integer",
                        "description": "Spark convention: 1=Sun, 2=Mon, \u2026, 7=Sat",
                    },
                    "is_weekend": {
                        "type": "integer",
                        "description": "1 if Saturday or Sunday, otherwise 0",
                    },
                    "passenger_count": {
                        "type": "integer",
                        "description": "Number of passengers (default 1)",
                    },
                    "rate_code_id": {
                        "type": "integer",
                        "description": "TLC rate code: 1=Standard, 2=JFK flat rate, 3=Newark (default 1)",
                    },
                },
                "required": [
                    "pickup_lat",
                    "pickup_lon",
                    "dropoff_lat",
                    "dropoff_lon",
                    "hour_of_day",
                    "day_of_week",
                    "is_weekend",
                ],
            },
        },
    },
]

_TOOL_DISPATCH = {
    "get_datetime_features": get_datetime_features,
    "geocode_address": geocode_address,
    "get_weather": get_weather,
    "predict_duration": predict_duration,
}

_SYSTEM_PROMPT = """You are a helpful NYC taxi trip duration assistant.

When a user asks how long a taxi journey will take, you MUST:
1. Call get_datetime_features() to get the current time context.
2. Call geocode_address() for the pickup location.
3. Call geocode_address() for the dropoff location.
4. Call get_weather() at the pickup coordinates.
5. Call predict_duration() with all collected inputs.
6. Respond in plain English with the estimated duration, estimated road distance, 
   current weather conditions, and a brief note on factors that may affect the journey 
   (e.g. rush hour, rain, weekend vs weekday).

Always use the tools — never guess coordinates, distances, or durations.
If a pickup or dropoff location is ambiguous, ask the user to clarify before calling tools."""


def ask_agent(user_input: str) -> str:
    """Run the trip duration agent with a natural-language query."""
    messages = [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {"role": "user", "content": user_input},
    ]

    for _ in range(10):  # max 10 tool-call rounds
        response = deploy_client.predict(
            endpoint=AGENT_LLM_ENDPOINT,
            inputs={
                "messages": messages,
                "tools": _TOOLS_SCHEMA,
                "tool_choice": "auto",
            },
        )
        msg = response.choices[0]["message"]

        if not msg.get("tool_calls"):
            return msg["content"]

        messages.append(msg)
        for tc in msg["tool_calls"]:
            fn_name = tc["function"]["name"]
            fn_args = json.loads(tc["function"]["arguments"])
            print(
                f"  \u2192 {fn_name}({', '.join(f'{k}={v!r}' for k, v in fn_args.items())})"
            )
            result = _TOOL_DISPATCH[fn_name](**fn_args)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc["id"],
                    "content": json.dumps(result),
                }
            )

    return "Agent did not reach a conclusion within the tool-call limit."


print("Agent ready.")
print(
    "Call: ask_agent('I am at Times Square and want a taxi to JFK. How long will it take?')"
)

## Demo

In [ ]:
query = "I'm at Times Square and want a taxi to Penn Station. How long will it take?"
print(f"User: {query}\n")
print("Tool calls:")
answer = ask_agent(query)
print(f"\nAgent: {answer}")

In [ ]:
# Try another query
query2 = (
    "How long would a yellow cab take from JFK airport to the Empire State Building?"
)
print(f"User: {query2}\n")
print("Tool calls:")
answer2 = ask_agent(query2)
print(f"\nAgent: {answer2}")